In [ ]:
# Datenaufbereitung: Merge SieBERT + VADER + Original
# Merge der Sentiment-Ergebnisse mit Originaldaten, Entfernen 3-Sterne, Binary Labels

import pandas as pd
from pathlib import Path

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"

original = pd.read_csv(PATH / 'All_Reviews_Organic.csv')
siebert  = pd.read_csv(PATH / 'reviews_full_siebert_results.csv')
vader    = pd.read_csv(PATH / 'reviews_vader_results.csv')

siebert_cols = siebert[['ID', 'Predicted_Sentiment', 'Confidence', 'Correct']].rename(columns={
    'Predicted_Sentiment': 'SieBERT_Sentiment',
    'Confidence':          'SieBERT_Confidence',
    'Correct':             'SieBERT_Correct'
})

vader_cols = vader[['ID', 'VADER_Sentiment', 'VADER_Compound', 'VADER_Correct']]

merged = original.merge(siebert_cols, on='ID', how='left')
merged = merged.merge(vader_cols,    on='ID', how='left')

def star_to_label(rating):
    if rating in [1, 2]:   return 'negative'
    elif rating in [4, 5]: return 'positive'
    else:                  return None

merged['Ground_Truth'] = merged['Rating'].apply(star_to_label)

before = len(merged)
merged = merged[merged['Ground_Truth'].notna()].reset_index(drop=True)
after  = len(merged)

merged.to_csv(PATH / 'reviews_combined_results.csv', index=False)

print("\u2705 Fertig! Datei gespeichert als: reviews_combined_results.csv")
print(f"   3-Sterne entfernt:    {before - after} Reviews")
print(f"   Verbleibende Reviews: {after}")
print(f"   \u2192 Positiv:  {(merged['Ground_Truth'] == 'positive').sum()}")
print(f"   \u2192 Negativ:  {(merged['Ground_Truth'] == 'negative').sum()}")
print(f"   Sustainability: {merged['Sustainability'].value_counts().to_dict()}")
print(f"   Gender:         {merged['Gender'].value_counts().to_dict()}")
print(f"   Brand:          {merged['Brand'].value_counts().to_dict()}")

In [ ]:
# Genutzt zum Erstellen von Tabelle 6: Metrics comparison SiEBERT vs. VADER (Kap. 4.1.1)

import pandas as pd
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"
df   = pd.read_csv(PATH / 'reviews_combined_results.csv')

y_true    = df['Ground_Truth']
y_siebert = df['SieBERT_Sentiment']
y_vader   = df['VADER_Sentiment']

for name, y_pred in [('SieBERT', y_siebert), ('VADER', y_vader)]:
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")

    print(f"\n  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")

    print(f"\n  [Macro Averaging]")
    print(f"  Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred,    average='macro'):.4f}")
    print(f"  F1-Score:  {f1_score(y_true, y_pred,        average='macro'):.4f}")

    print(f"\n  [Detailed Report per Class]")
    print(classification_report(y_true, y_pred,
                                target_names=['negative', 'positive']))

In [ ]:
# Accuracy by Review Length (Kap. 4.1.1)

import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"
df   = pd.read_csv(PATH / 'reviews_combined_results.csv')

# Word count on review body (Text), as defined in Chapter 3.1
df['word_count'] = df['Text'].fillna('').astype(str).str.split().str.len()

# Split at 14 words (median) and 25 words (Q3), as used in Kap. 3.1
short = df[df['word_count'] < 14]
long  = df[df['word_count'] > 25]

y_true = 'Ground_Truth'

print(f"{'='*60}")
print(f"  Accuracy by Review Length")
print(f"{'='*60}")

for name, col in [('SieBERT', 'SieBERT_Sentiment'), ('VADER', 'VADER_Sentiment')]:
    acc_short = accuracy_score(short[y_true], short[col])
    acc_long  = accuracy_score(long[y_true],  long[col])
    print(f"\n  {name}:")
    print(f"    Short (<14 words, n={len(short)}):  {acc_short:.4f}")
    print(f"    Long  (>25 words, n={len(long)}):   {acc_long:.4f}")
    print(f"    Difference:                         {acc_short - acc_long:+.4f}")

In [ ]:
# Tabelle 8: Robustness Check - Full vs. Filtered Dataset (Kap. 4.1.3)

import pandas as pd
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"

# Filtered (organic) dataset
df_filtered = pd.read_csv(PATH / 'reviews_combined_results.csv')

# Full dataset (including incentivized) - 3-star already excluded, ground truth in 'Sentiment'
df_full = pd.read_csv(PATH / 'Robust_SA_All.csv')

# Ground truth is in 'Sentiment' column (3-star reviews already excluded)
df_full['Ground_Truth'] = df_full['Sentiment']

print(f"Filtered dataset: n={len(df_filtered)}")
print(f"Full dataset:     n={len(df_full)}")

def compute_metrics(y_true, y_pred, label):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec  = recall_score(y_true, y_pred,    average='macro')
    f1   = f1_score(y_true, y_pred,        average='macro')
    print(f"  {label:25s}  Acc: {acc:.4f}  Prec: {prec:.4f}  Rec: {rec:.4f}  F1: {f1:.4f}")

print(f"\n{'='*75}")
print(f"  Table 8: Metrics Comparison Full Dataset vs. Filtered")
print(f"{'='*75}")

for name, col in [('SieBERT', 'SieBERT_Sentiment'), ('VADER', 'VADER_Sentiment')]:
    print(f"\n  {name}:")
    compute_metrics(df_filtered['Ground_Truth'], df_filtered[col], f"{name} (filtered)")
    compute_metrics(df_full['Ground_Truth'],     df_full[col],     f"{name} (all)")

In [ ]:
# Tabelle 9 (Sentiment by Brand) und Tabelle 10 (H3: Sustainable by Brand)

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

path = str(__import__("pathlib").Path.home() / "Desktop" / "Masterarbeit" / "SA" / "reviews_combined_results.csv")
df = pd.read_csv(path)
print(f"Total reviews: {len(df)}\n")


def chi_square(ct, label):
    """Chi-square test with Cram\u00e9r's V"""
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n  {label}")
    print(f"  {'-'*55}")
    for idx in ct.index:
        total = ct.loc[idx].sum()
        pos = ct.loc[idx, 'positive']
        neg = ct.loc[idx, 'negative']
        print(f"    {str(idx):15s}  n={total:,}  |  pos {pos:,} ({pos/total*100:.1f}%)  |  neg {neg:,} ({neg/total*100:.1f}%)")
    print(f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.2e}" if p < 0.001 else f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.6f}")
    print(f"    Cram\u00e9r's V = {v:.4f}")


print("=" * 60)
print("  Table 9: OVERALL Nike vs. Adidas")
print("=" * 60)
ct_all = pd.crosstab(df['Brand'], df['SieBERT_Sentiment'])
chi_square(ct_all, "All reviews")

print(f"\n\n{'='*60}")
print("  Table 10 / H3: Nike sust. vs. Adidas sust. (eco-communication)")
print("=" * 60)
sust = df[df['Sustainability'] == 'Sustainable']
ct_h3 = pd.crosstab(sust['Brand'], sust['SieBERT_Sentiment'])
chi_square(ct_h3, "Sustainable only")

print(f"\n\n{'='*60}")
print("  CONTEXT: Conventional by brand")
print("=" * 60)
conv = df[df['Sustainability'] == 'Conventional']
ct_conv = pd.crosstab(conv['Brand'], conv['SieBERT_Sentiment'])
chi_square(ct_conv, "Conventional only")

print("\n\nDone.")

In [ ]:
# Tabelle 11: H1 Sustainable vs. Conventional (Kap. 4.3)

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

path = str(__import__("pathlib").Path.home() / "Desktop" / "Masterarbeit" / "SA" / "reviews_combined_results.csv")
df = pd.read_csv(path)
print(f"Total reviews: {len(df)}\n")


def chi_square(ct, label):
    """Chi-square test with Cram\u00e9r's V"""
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"\n  {label}")
    print(f"  {'-'*55}")
    for idx in ct.index:
        total = ct.loc[idx].sum()
        pos = ct.loc[idx, 'positive']
        neg = ct.loc[idx, 'negative']
        print(f"    {str(idx):15s}  n={total:,}  |  pos {pos:,} ({pos/total*100:.1f}%)  |  neg {neg:,} ({neg/total*100:.1f}%)")
    print(f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.2e}" if p < 0.001 else f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.6f}")
    print(f"    Cram\u00e9r's V = {v:.4f}")

print("=" * 60)
print("  Table 11 / H1: Sustainable vs. Conventional (overall)")
print("=" * 60)
ct_h1 = pd.crosstab(df['Sustainability'], df['SieBERT_Sentiment'])
chi_square(ct_h1, "All reviews")

for brand in ['Nike', 'Adidas']:
    print(f"\n\n{'='*60}")
    print(f"  H1 within {brand}")
    print("=" * 60)
    sub = df[df['Brand'] == brand]
    ct_b = pd.crosstab(sub['Sustainability'], sub['SieBERT_Sentiment'])
    chi_square(ct_b, f"{brand} only")


print("\n\nDone.")

In [ ]:
# Tabelle 12: H2 Gender-Vergleich (Kap. 4.4)

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"
path = PATH / "reviews_combined_results.csv"
df = pd.read_csv(path)
print(f"Total reviews: {len(df)}\n")

def chi_square(ct, label):
    """Chi-square test with Cram\u00e9r's V"""
    print(f"\n  {label}")
    print(f"  {'-'*55}")
    for idx in ct.index:
        pos = ct.loc[idx, 'POSITIVE']
        neg = ct.loc[idx, 'NEGATIVE']
        total = pos + neg
        print(f"    {str(idx):15s}  n={total:,}  |  pos {pos:,} ({pos/total*100:.1f}%)  |  neg {neg:,} ({neg/total*100:.1f}%)")
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.2e}" if p < 0.001 else f"    \u03c7\u00b2({dof}) = {chi2:.4f},  p = {p:.6f}")
    print(f"    Cram\u00e9r's V = {v:.4f}")

print("=" * 60)
print("  Table 12 / H2: Women's vs. Men's sustainable products (overall)")
print("=" * 60)
sust = df[df['Sustainability'] == 'Sustainable']
ct_h2 = pd.crosstab(sust['Gender'], sust['SieBERT_Sentiment'])
chi_square(ct_h2, "Sustainable by Gender (overall)")

for brand in ['Nike', 'Adidas']:
    print(f"\n\n{'='*60}")
    print(f"  H2 within {brand}")
    print("=" * 60)
    sub = sust[sust['Brand'] == brand]
    ct = pd.crosstab(sub['Gender'], sub['SieBERT_Sentiment'])
    chi_square(ct, f"Sustainable by Gender ({brand})")

print("\n\nDone.")

In [ ]:
# Zusatzanalyse Diskussion (Kap. 5.1): Welch's t-Test auf Star Ratings fuer H2

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import ttest_ind

PATH = Path.home() / "Desktop" / "Masterarbeit" / "SA"
df = pd.read_csv(PATH / 'reviews_combined_results.csv')

sust = df[df['Sustainability'] == 'Sustainable']

women_ratings = sust[sust['Gender'] == 'Female']['Rating']
men_ratings   = sust[sust['Gender'] == 'Male']['Rating']

# Welch's t-test (equal_var=False)
t_stat, p_value = ttest_ind(women_ratings, men_ratings, equal_var=False)

# Cohen's d
mean_diff = women_ratings.mean() - men_ratings.mean()
pooled_std = np.sqrt((women_ratings.std()**2 + men_ratings.std()**2) / 2)
cohens_d = mean_diff / pooled_std

print(f"{'='*60}")
print(f"  Welch's t-Test: Star Ratings Women vs. Men (Sustainable)")
print(f"{'='*60}")
print(f"\n  Women's sustainable (n={len(women_ratings)}):  Mean={women_ratings.mean():.4f}, SD={women_ratings.std():.4f}")
print(f"  Men's sustainable   (n={len(men_ratings)}):  Mean={men_ratings.mean():.4f}, SD={men_ratings.std():.4f}")
print(f"\n  Mean difference: {mean_diff:.4f}")
print(f"  t-statistic:     {t_stat:.4f}")
print(f"  p-value:         {p_value:.2e}" if p_value < 0.001 else f"  p-value:         {p_value:.6f}")
print(f"  Cohen's d:       {cohens_d:.4f}")
print(f"\n  Interpretation: {'Significant' if p_value < 0.05 else 'Not significant'} at alpha=0.05")